# V80 — YOLO ∪ component-tree 候補生成: fold2

YOLO OOF キャッシュ + V34 semantic 確率マップから候補を作り、fold2 専用
refiner (`mediseg_refiner_fold2_resume.pt`) を shrink@0.45 で適用して
`<MyDrive>/solar_filament_v80/candidates/fold_2/` に保存する。
実行するのは Drive 上の `solar_filament_v80/repo` スナップショットの
`run_v80_hybrid_candidates.py` そのもの(コード複製なし)。

**前提**: ローカル側で `Solar_Filament_Segmentation/{solar_filament,scratch,src}` を
`G:\マイドライブ\solar_filament_v80\repo\` にコピー済みであること。

GPU必須 (T4 で fold あたり 1〜2h 見込み)。


In [ ]:
# [1/3] Drive接続・前提確認
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/drive/MyDrive/solar_filament_v80/repo")
SCRIPT = REPO / "scratch" / "run_v80_hybrid_candidates.py"
for required in (SCRIPT, REPO / "solar_filament", REPO / "src"):
    if not required.exists():
        raise FileNotFoundError(
            f"{required} がありません。ローカルから Drive へ repo スナップショットを"
            "再コピーしてください (solar_filament/ scratch/ src/ の3ディレクトリ)"
        )
sys.path.insert(0, str(REPO))
!pip -q install pycocotools
import torch
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({"GPU": torch.cuda.get_device_name(0), "repo": str(REPO)})


In [ ]:
# [2/3] fold2 の候補生成 (GPU)
import importlib.util

spec = importlib.util.spec_from_file_location("v80_candidates", SCRIPT)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)  # __name__ != "__main__" なので main() は走らない

sys.argv = [
    "run_v80_hybrid_candidates.py",
    "--fold", "2",
    "--threads", "2",           # CPU 部分 (cv2/tree) のみ。refiner は GPU
    "--skip-existing",          # 中断再開に対応
]
mod.main()


In [ ]:
# [3/3] manifest 確認
import json

manifest = json.loads(
    (REPO.parent / "candidates" / "manifest_fold2.json")
    .read_text(encoding="utf-8"))
manifest.pop("input_fingerprints", None)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
